In [ ]:
# Pagination example

In [120]:
import requests
import json
import time
from datetime import datetime
import logging

URL = "https://rickandmortyapi.com/api/character"
OUTPUT_FILE = "../output/rick_morty.jsonl"

logging.basicConfig(level=logging.INFO)

In [121]:
def extract_character(start_url):
    next_url = start_url
    page_count = 1
    max_page_count = 10

    while next_url and page_count <= max_page_count:
        logging.info(f"Extracting page {page_count}: {next_url}")
        try:
            response = requests.get(next_url, timeout=(5,15))
            response.raise_for_status()

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After"))
                logging.info(f"Rate limited, sleeping for {retry_after} seconds")
                time.sleep(retry_after)
                continue

            payload = response.json()
        except json.JSONDecodeError:
            logging.error(f"Failed to parse JSON on page {page_count}")
            break
        except Exception as e:
            logging.error(f"Exception: {e}")
            break

        characters = payload.get('results',[])
        for char in characters:
            yield char
        
        # if I'm just adding all the data to a list instead of yielding a single character
        # all_data.extend(data['results'])
        #    page += 1

        next_url = payload.get('info', {}).get('next')
        page_count += 1

        time.sleep(1)

In [122]:
def transform_character(char):
    curr_time = datetime.now()
    return {
        "character_id": char.get("id"),
        "name": char.get("name"),
        "gender": char.get("gender"),
        "origin_name": char.get("origin").get("name"),
        "extracted_at": curr_time.strftime("%Y-%m-%dT%H:%M:%SZ")
    }

In [124]:
def run_pipeline():
    logging.info("Starting ETL pipeline")
    record_count = 0
    with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
        for record in extract_character(URL):
            transformed_record = transform_character(record)

            file.write(json.dumps(transformed_record) + "\n")
            record_count += 1
    
    logging.info(f"Pipeline complete, wrote {record_count} records")

In [125]:
run_pipeline()

INFO:root:Starting ETL pipeline
INFO:root:Extracting page 1: https://rickandmortyapi.com/api/character
INFO:root:Extracting page 2: https://rickandmortyapi.com/api/character?page=2
INFO:root:Extracting page 3: https://rickandmortyapi.com/api/character?page=3
INFO:root:Extracting page 4: https://rickandmortyapi.com/api/character?page=4
INFO:root:Extracting page 5: https://rickandmortyapi.com/api/character?page=5
INFO:root:Extracting page 6: https://rickandmortyapi.com/api/character?page=6
INFO:root:Extracting page 7: https://rickandmortyapi.com/api/character?page=7
INFO:root:Extracting page 8: https://rickandmortyapi.com/api/character?page=8
INFO:root:Extracting page 9: https://rickandmortyapi.com/api/character?page=9
INFO:root:Extracting page 10: https://rickandmortyapi.com/api/character?page=10
INFO:root:Pipeline complete, wrote 200 records


In [ ]:
# Streaming

In [112]:
import csv
import json
import logging
import requests

logging.basicConfig(level=logging.INFO)

DATA_URL = "https://httpbin.org/"
OUTPUT_FILE = "../output/etl_output.csv"

In [113]:
def extract_stream(url):
    logging.info(f"Starting extraction from {url}")

    try:
        with requests.get(url, stream=True, timeout=30) as response:
            response.raise_for_status()

            for raw_line in response.iter_lines(decode_unicode=True):
                if raw_line:
                    yield raw_line

    except Exception as e:
        logging.error(f"Error during extraction: {e}")
        return

In [119]:
def transform_record(raw_line):
    logging.info("Starting transformation")
    try:
        data = json.loads(raw_line)

        transformed = {
            "record_id": data.get("id"),
            "source_ip": data.get("source_ip"),
            "user_agent": data.get("headers").get("User-Agent", "Unknown"),
            "status": "PROCESSED" if data.get("id") else "FLAGGED"
        }
        return transformed
    
    except (json.JSONDecodeError, TypeError) as e:
        logging.warning(f"Skipping malformed row due to JSON error: {e}")
        return None
    except Exception as e:
        logging.warning(f"Other error: {e}")
        return None

In [115]:
def load_to_csv(generator, output_path):
    logging.info(f"Beginning load phase to {output_path}")

    csv_headers = ["record_id", "source_ip", "user_agent", "status"]
    records_written = 0

    try:
        with open(output_path, mode="w", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=csv_headers)
            writer.writeheader()

            for raw_line in generator:
                transformed_data = transform_record(raw_line)

                if transformed_data:
                    writer.writerow(transformed_data)
                    records_written += 1
        
        logging.info(f"ETL job complete, loaded {records_written} records")

    except Exception as e:
        logging.info(f"Error during processing: {e}")

In [118]:
raw_stream = extract_stream(DATA_URL)
load_to_csv(raw_stream, OUTPUT_FILE)

INFO:root:Beginning load phase to ../output/etl_output.csv
INFO:root:Starting extraction from https://httpbin.org/
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
INFO:root:Starting transformation
I